# 05 — Evaluación final, negocio y registro
SVR queda como **champion** y XGBoost como **challenger** por la selección previa basada en CV. Su diferencia de RMSE es mucho menor que la dispersión entre folds: no demuestra superioridad. No se escogerán parámetros, features, umbrales ni otro ganador a partir de test.

Escenario aprobado: ordenar por pKd predicho y seleccionar el **20 %** de mayor valor, con cupo `floor(0.20 × N)`. Un acierto de negocio es un seleccionado con **pKd experimental ≥ 7** (Kd ≤ 100 nM). No se exige que la predicción supere 7 para completar el cupo; es una selección de presupuesto fijo. La precisión y la recuperación informan qué conservamos y qué perdemos.

El umbral es ilustrativo, inspirado en el rango 10–100 nM descrito para determinados anticuerpos en [mAbs, 2020](https://pmc.ncbi.nlm.nih.gov/articles/PMC7531523/); no es criterio universal de eficacia. La simulación aplica a este dataset heterogéneo y no prueba rendimiento en una campaña real.

La tarifa de [UIC](https://bpc.uic.edu/instruments-services/services/) es **US$402,86 por muestra ITC con control de buffer** y presupone **7 horas de instrumento**. Es tarifa interna de referencia, no cotización local. Se supone proporcionalidad por muestra, estructuras y proteínas disponibles, sin incluir producción, cristalografía ni cambios de protocolo. Las horas liberadas no son tiempo calendario.

El ahorro bruto compara medir todos contra medir el cupo: será igual para ambos modelos y para selección aleatoria. Lo que distingue su valor es cuántos candidatos de interés recuperan. Se muestra también un escenario neto ilustrativo restando US$100 por lote de operación ML, supuesto no medido. No se afirma sustitución de evidencia experimental.


In [1]:
from pathlib import Path
import sys, json, hashlib, math, time
import numpy as np
import pandas as pd
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
import mlflow
from mlflow import MlflowClient
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.tracking import configurar_tracking
from threadpoolctl import threadpool_limits
snapshot=Path(json.loads((ROOT/"configs/dataset.local.json").read_text())["snapshot"]).name
base=ROOT/"data/modelado"/snapshot
salida=ROOT/"reports/final"/snapshot
salida.mkdir(parents=True,exist_ok=True)
tuning=pd.read_csv(ROOT/"reports/tuning"/snapshot/"comparacion_tuning.csv").set_index("modelo")
split_manifest=json.loads((base/"split/manifest.json").read_text())
manifest=json.loads((base/"manifest.json").read_text())
politica={
    "primary_metric":"RMSE", "secondary_metrics":["MAE","R2"],
    "champion":"SVR","challenger":"XGBRegressor", "target_threshold":7.0,
    "selection_fraction":0.20,"rounding":"floor","selection_rule":"top predictions, no predicted cutoff",
    "random_seed":42,"random_repetitions":10000,"itc_usd_per_measurement":402.86,
    "itc_instrument_hours_per_measurement":7.0,"illustrative_ml_lot_cost_usd":100.0,
    "cost_source":"https://bpc.uic.edu/instruments-services/services/",
    "threshold_source":"https://pmc.ncbi.nlm.nih.gov/articles/PMC7531523/",
    "source_sha256":split_manifest["source_sha256"],
    "split_assignment_sha256":split_manifest["assignment_sha256"],
    "model_uris":{m:tuning.loc[m,"pipeline_uri"] for m in ["SVR","XGBRegressor"]},
    "registry_name":"protein-affinity-modelo",
}
politica_archivo=ROOT/"configs/evaluacion_final.json"
if politica_archivo.exists():
    assert json.loads(politica_archivo.read_text())==politica, "Protocolo congelado: no cambiar tras evaluar test"
else:
    politica_archivo.write_text(json.dumps(politica,indent=2))
politica_hash=hashlib.sha256(politica_archivo.read_bytes()).hexdigest()
configurar_tracking()
print("Protocolo congelado antes de evaluación:",politica_hash)


Protocolo congelado antes de evaluación: a99cbf3fe3c53d785c2175cf2ea138c7f51f96765f0e64cb7a070522583882f5


## 1. Cargar únicamente la partición congelada
La primera ejecución genera predicciones de test para los dos pipelines ya ajustados en entrenamiento. Las siguientes reutilizan las predicciones guardadas y comprueban hashes; no repiten selección ni tuning.


In [2]:
for archivo,esperado in [(base/"datos_modelado.parquet",politica["source_sha256"]),
                         (base/"split/asignacion.parquet",politica["split_assignment_sha256"])]:
    assert hashlib.sha256(archivo.read_bytes()).hexdigest()==esperado
datos=pd.read_parquet(base/"datos_modelado.parquet")
asignacion=pd.read_parquet(base/"split/asignacion.parquet")
assert datos.index.equals(asignacion.index)
test=datos.loc[asignacion.PARTICION.eq("test")]
train=datos.loc[asignacion.PARTICION.eq("train")]
assert not set(train[manifest["group"]]) & set(test[manifest["group"]])
assert not asignacion.loc[test.index,"ENTRENAMIENTO_OBLIGATORIO"].any()
X_test=test[manifest["predictors"]]
y_test=test[manifest["target"]]
N=len(test);K=math.floor(N*politica["selection_fraction"])
assert 0<K<N
print("Test:",N,"| Cupo:",K,"| Porcentaje efectivo:",round(100*K/N,2))


Test: 777 | Cupo: 155 | Porcentaje efectivo: 19.95


## 2. Métricas técnicas y decisión económica
Los casos no seleccionados no se declaran malos: se posponen. La selección aleatoria usa el mismo cupo y se simula con una distribución hipergeométrica, equivalente a muestrear sin reemplazo 10 000 veces. Sus percentiles son dispersión de la referencia aleatoria, no un intervalo de confianza del modelo. No imputamos afinidades.


In [3]:
resultados=[]
positivos=y_test.ge(politica["target_threshold"])
P=int(positivos.sum())
c=politica["itc_usd_per_measurement"]
rng=np.random.default_rng(politica["random_seed"])
hits_random=rng.hypergeometric(P,N-P,K,size=politica["random_repetitions"])
referencia={"N":N,"K":K,"positivos_disponibles":P,"aciertos_esperados":K*P/N,
            "aciertos_promedio_simulado":float(hits_random.mean()),
            "percentil_2_5":float(np.quantile(hits_random,0.025)),"percentil_97_5":float(np.quantile(hits_random,0.975)),
            "costo_medir_todos_usd":N*c,"costo_medir_cupo_usd":K*c,
            "ahorro_bruto_escenario_usd":(N-K)*c,
            "horas_instrumento_liberadas":(N-K)*politica["itc_instrument_hours_per_measurement"]}
for nombre,uri in politica["model_uris"].items():
    archivo=salida/f"predicciones_{nombre}.parquet"
    metadata=salida/f"predicciones_{nombre}.json"
    if archivo.exists():
        meta=json.loads(metadata.read_text())
        assert meta["policy_sha256"]==politica_hash and meta["model_uri"]==uri
        assert hashlib.sha256(archivo.read_bytes()).hexdigest()==meta["prediction_sha256"]
        predicciones=pd.read_parquet(archivo)
        assert predicciones.index.equals(X_test.index)
        np.testing.assert_array_equal(predicciones.pkd_real.to_numpy(),y_test.to_numpy())
    else:
        modelo=mlflow.sklearn.load_model(uri)
        with threadpool_limits(limits=1,user_api="blas"):
            inicio=time.perf_counter();pred=modelo.predict(X_test);segundos=time.perf_counter()-inicio
        assert np.isfinite(pred).all()
        predicciones=pd.DataFrame({"pkd_real":y_test,"pkd_predicho":pred},index=test.index)
        predicciones.to_parquet(archivo,index=True)
        meta={"policy_sha256":politica_hash,"model_uri":uri,"inference_seconds":segundos,
              "prediction_sha256":hashlib.sha256(archivo.read_bytes()).hexdigest()}
        metadata.write_text(json.dumps(meta,indent=2))
    pred=predicciones.pkd_predicho
    # Desempate estable por índice original, sin mirar afinidad experimental.
    orden=predicciones.sort_index().sort_values("pkd_predicho",ascending=False,kind="stable")
    seleccion=orden.head(K)
    hits=int(seleccion.pkd_real.ge(politica["target_threshold"]).sum())
    fila={"modelo":nombre,"rmse":float(np.sqrt(mean_squared_error(y_test,pred))),
          "mae":float(mean_absolute_error(y_test,pred)),"r2":float(r2_score(y_test,pred)),
          "seleccionados":K,"aciertos":hits,"precision_seleccion":hits/K,
          "recuperacion":hits/P if P else None,"prometedores_no_seleccionados":P-hits,
          "seleccionados_bajo_umbral":K-hits,
          "aciertos_extra_vs_azar":hits-referencia["aciertos_esperados"],
          "costo_experimental_por_acierto_usd":K*c/hits if hits else None,
          "ahorro_bruto_escenario_usd":referencia["ahorro_bruto_escenario_usd"],
          "ahorro_neto_ilustrativo_usd":referencia["ahorro_bruto_escenario_usd"]-politica["illustrative_ml_lot_cost_usd"],
          "inference_seconds":meta["inference_seconds"]}
    run_archivo=salida/f"run_{nombre}.json"
    if run_archivo.exists():
        registro=json.loads(run_archivo.read_text());assert registro["policy_sha256"]==politica_hash
        run_id=registro["run_id"]
    else:
        with mlflow.start_run(run_name=f"final_{nombre}") as run:
            run_id=run.info.run_id
            mlflow.set_tags({"phase":"final","family":nombre,"test_used":"true","selection_basis":"frozen_CV_not_test"})
            mlflow.log_dict(politica,"politica.json");mlflow.log_dict(fila,"resultados.json")
            mlflow.log_params({"model_uri":uri,"threshold":7.0,"budget_fraction":0.2,"N":N,"K":K})
            mlflow.log_metrics({k:float(v) for k,v in fila.items() if isinstance(v,(int,float))})
            mlflow.log_artifact(str(archivo))
        run_archivo.write_text(json.dumps({"run_id":run_id,"policy_sha256":politica_hash},indent=2))
    fila["run_id"]=run_id;resultados.append(fila)
resumen_final=pd.DataFrame(resultados)
resumen_final.to_csv(salida/"metricas_finales.csv",index=False)
(salida/"referencia_aleatoria.json").write_text(json.dumps(referencia,indent=2))
# Registrar la referencia de negocio sin atribuirle ahorro por capacidad predictiva.
if not (salida/"run_azar.json").exists():
    with mlflow.start_run(run_name="business_random_selection") as r:
        mlflow.set_tags({"phase":"business_baseline","test_used":"true"})
        mlflow.log_dict(referencia,"referencia.json");mlflow.log_dict(politica,"politica.json")
        mlflow.log_metrics({k:float(v) for k,v in referencia.items()})
    (salida/"run_azar.json").write_text(json.dumps({"run_id":r.info.run_id,"policy_sha256":politica_hash}))
display(resumen_final.round(4))
display(pd.DataFrame([referencia]).round(2))


2026/11/19 04:04:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run final_SVR at: http://mlflow:5000/#/experiments/1/runs/188dd3f5fb7d4392be0a17e37a5b0ac5.


2026/11/19 04:04:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


2026/11/19 04:04:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run final_XGBRegressor at: http://mlflow:5000/#/experiments/1/runs/e5c3d9ba7e3641869d762bf5d8c3113e.


2026/11/19 04:04:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


2026/11/19 04:04:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run business_random_selection at: http://mlflow:5000/#/experiments/1/runs/cd1be1a19c544702842bd39286baab80.


2026/11/19 04:04:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


,modelo,rmse,mae,r2,seleccionados,aciertos,precision_seleccion,recuperacion,prometedores_no_seleccionados,seleccionados_bajo_umbral,aciertos_extra_vs_azar,costo_experimental_por_acierto_usd,ahorro_bruto_escenario_usd,ahorro_neto_ilustrativo_usd,inference_seconds,run_id
0,SVR,1.5847,1.2351,0.2001,155,131,0.8452,0.2770,342,24,36.6435,476.6664,250578.92,250478.92,1.0064,188dd3f5fb7d4392be0a17e37a5b0ac5
1,XGBRegressor,1.5879,1.2286,0.1968,155,137,0.8839,0.2896,336,18,42.6435,455.7905,250578.92,250478.92,0.0108,e5c3d9ba7e3641869d762bf5d8c3113e


,N,K,positivos_disponibles,aciertos_esperados,aciertos_promedio_simulado,percentil_2_5,percentil_97_5,costo_medir_todos_usd,costo_medir_cupo_usd,ahorro_bruto_escenario_usd,horas_instrumento_liberadas
0,777,155,473,94.36,94.26,83.0,105.0,313022.22,62443.3,250578.92,4354.0


## 3. Registry y decisión congelada
Se conservan los roles acordados antes de test: SVR champion y XGBoost challenger. El alias champion significa elección operativa del proyecto, no superioridad estadísticamente demostrada. Los pipelines registrados incluyen preparación y estimador ya ajustados sobre entrenamiento; no se reajustan con test.

En costo de entrenamiento distinguimos tiempo de búsqueda de tiempo de ajuste del pipeline ganador. La latencia aquí es una medición local por lote de 777 filas, sin incluir carga del modelo, extracción estructural ni red. SVR con kernel RBF y XGBoost no son modelos intrínsecamente transparentes; XGBoost permite inspeccionar importancias, pero no son explicación causal. No se calculan importancias sobre test para volver a seleccionar features.


In [4]:
cliente=MlflowClient()
registro_archivo=salida/"registry.json"
if registro_archivo.exists():
    registro=json.loads(registro_archivo.read_text());assert registro["policy_sha256"]==politica_hash
else:
    registro={"policy_sha256":politica_hash,"name":politica["registry_name"],"models":{}}
    for alias in ["champion","challenger"]:
        nombre=politica[alias];uri=politica["model_uris"][nombre]
        version=mlflow.register_model(uri,politica["registry_name"])
        cliente.set_model_version_tag(politica["registry_name"],version.version,"policy_sha256",politica_hash)
        cliente.set_model_version_tag(politica["registry_name"],version.version,"family",nombre)
        cliente.set_registered_model_alias(politica["registry_name"],alias,version.version)
        registro["models"][alias]={"family":nombre,"version":str(version.version),"source_uri":uri}
    registro_archivo.write_text(json.dumps(registro,indent=2))
for alias,info in registro["models"].items():
    assert str(cliente.get_model_version_by_alias(registro["name"],alias).version)==info["version"]
display(pd.DataFrame(registro["models"]).T)


Successfully registered model 'protein-affinity-modelo'.
2026/11/19 04:04:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: protein-affinity-modelo, version 1


Created version '1' of model 'protein-affinity-modelo'.
Registered model 'protein-affinity-modelo' already exists. Creating a new version of this model...


2026/11/19 04:04:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: protein-affinity-modelo, version 2


Created version '2' of model 'protein-affinity-modelo'.


,family,version,source_uri
champion,SVR,1,runs:/72fe83523ad74823a97ba08087442c50/pipeline
challenger,XGBRegressor,2,runs:/ad23c738cbcb43a488fb85ed7db2dc4b/pipeline


## 4. Reproducción en un proceso Python limpio y exportación
La comprobación recarga el champion desde el Registry en un proceso nuevo. Reproduce las mismas predicciones y RMSE ya congelados; no es una nueva selección ni consulta para ajustar el modelo.


In [5]:
import subprocess
verificacion = r"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import mlflow
from sklearn.metrics import mean_squared_error
from src.tracking import configurar_tracking
configurar_tracking()
p=Path('configs/evaluacion_final.json');pol=json.loads(p.read_text())
snapshot=Path(json.loads(Path('configs/dataset.local.json').read_text())['snapshot']).name
base=Path('data/modelado')/snapshot
out=Path('reports/final')/snapshot
m=json.loads((base/'manifest.json').read_text())
datos=pd.read_parquet(base/'datos_modelado.parquet')
pred=pd.read_parquet(out/f"predicciones_{pol['champion']}.parquet")
modelo=mlflow.sklearn.load_model(f"models:/{pol['registry_name']}@champion")
p=modelo.predict(datos.loc[pred.index,m['predictors']])
np.testing.assert_allclose(p,pred.pkd_predicho,rtol=1e-12,atol=1e-12)
rmse=float(np.sqrt(mean_squared_error(pred.pkd_real,p)))
esperado=pd.read_csv(out/'metricas_finales.csv').set_index('modelo').loc[pol['champion'],'rmse']
assert np.isclose(rmse,esperado,rtol=1e-12,atol=1e-12)
print(json.dumps({'verified':True,'rmse':rmse,'source':'registry_champion_clean_process'}))
"""
r=subprocess.run([sys.executable,"-c",verificacion],cwd=ROOT,text=True,capture_output=True,check=True)
print(r.stdout)
(salida/"reproduccion.json").write_text(r.stdout)
experimento=mlflow.get_experiment_by_name("protein-affinity")
corridas=mlflow.search_runs(experiment_ids=[experimento.experiment_id],max_results=10000)
corridas.to_csv(ROOT/"docs/corridas.csv",index=False)
print("Corridas exportadas:",len(corridas))


{"verified": true, "rmse": 1.5846640384508475, "source": "registry_champion_clean_process"}



Corridas exportadas: 112


## 5. Limitaciones y cierre
- Evaluación retrospectiva: la reducción de gasto y horas es un escenario de ensayos pospuestos, no ahorro observado. Con cupo fijo el ahorro no distingue modelos; sí lo hacen los aciertos y prometedores no seleccionados.
- Entrenamiento incluye 293 casos cuya similitud estructural no está representada; no se ha demostrado independencia respecto de test para esos casos.
- El dataset contiene complejos estudiados y medidos, no una muestra de todos los pares posibles ni evidencia de capacidad universal para detectar ausencia de unión.
- La búsqueda no obtuvo reducciones de RMSE superiores a la variación entre folds. Los roles de Registry no implican significancia estadística.
- Las dos features derivadas se evaluaron con Ridge en entrenamiento; no se incorporaron a los finalistas porque no mostraron una mejora superior a la variación observada.
- **LLM por prompting:** no se evalúa como regresor en esta entrega. No contamos con un modelo fundacional validado para convertir estos 173 resúmenes geométricos en pKd; convertirlos a texto no establece competencia ni una evaluación reproducible sin fijar modelo, versión y protocolo. Se documenta como alternativa no implementada, no como imposibilidad general.
- El comparador del artículo PPB-Affinity no se reprodujo; el baseline técnico es DummyMean y el de negocio es selección aleatoria, que no representa el procedimiento real de un laboratorio.


## Resultado observado y decisión
En 777 observaciones de test, SVR obtuvo RMSE **1,5847** y XGBoost **1,5879**. Se mantienen los roles congelados por CV: SVR champion (v1) y XGBoost challenger (v2). La diferencia técnica es pequeña y no prueba superioridad.

Con 155 selecciones y umbral pKd ≥ 7, SVR recuperó **131** positivos (84,5 % de los seleccionados) y XGBoost **137** (88,4 %), frente a **94,36 esperados al azar**. XGBoost obtuvo seis aciertos más en esta evaluación económica, aunque no se cambia el criterio retrospectivamente.

Había **473** positivos en test: SVR dejó sin seleccionar 342 y XGBoost 336. La recuperación es solo 27,7 % y 29,0 %. La selección concentra candidatos de interés, pero no preserva la mayoría de los positivos; por tanto, no justifica eliminar definitivamente los demás.

El escenario pospone 622 mediciones: **US$250 578,92 de costo experimental bruto y 4 354 horas de instrumento**, usando los supuestos publicados y lineales del protocolo. Es idéntico con selección aleatoria y no representa ahorro causal atribuible al modelo. El aporte del modelo se observa en los aciertos y costo por acierto, no en elegir arbitrariamente un cupo menor.

Los mejores pipelines están registrados y la reproducción del champion en un proceso limpio confirmó el mismo RMSE. No se realizaron nuevos ajustes tras consultar test.
